# CUAD Preprocessing Pipeline

This notebook covers the 6 steps to preprocess the CUAD dataset.

## STEP 1: Load the dataset
- Load CUADv1.json from data/raw/

In [34]:
import json
import re
import unicodedata
import pandas as pd
from pathlib import Path

# Set up paths
project_root = Path.cwd()
if project_root.name.lower() == "preprocessing":
    project_root = project_root.parents[1]

input_path = project_root / "data" / "raw" / "CUADv1.json"
output_path = project_root / "data" / "processed" / "clauses.csv"

# Load JSON
with open(input_path, "r", encoding="utf-8") as f:
    dataset = json.load(f)

print(f"Loaded dataset from {input_path}")

Loaded dataset from d:\Smart_Contract_Review_System\data\raw\CUADv1.json


In [35]:
MIN_WORDS = 8
MIN_CHAR_LEN = 20
MAX_CHAR_LEN = 2000
HTML_PATTERN = re.compile(r"<[^>]+>")
WHITESPACE_PATTERN = re.compile(r"\s+")

def normalize_clause_text(raw_text: str) -> str:
    """Normalize unicode, drop simple HTML, and collapse whitespace."""
    normalized = unicodedata.normalize("NFKC", raw_text)
    normalized = HTML_PATTERN.sub(" ", normalized)
    normalized = WHITESPACE_PATTERN.sub(" ", normalized)
    return normalized.strip()

## STEP 2 & STEP 3: Extract and Clean
- Extract clause text from answers["text"]
- Extract label from question using regex (text inside quotes "")
- Convert label to lowercase
- Remove empty texts
- Remove very short texts (less than 5 words)
- Strip whitespace
- Remove duplicates

In [36]:
cleaned_data = []
seen = set()

for contract in dataset.get("data", []):
    for paragraph in contract.get("paragraphs", []):
        for qa in paragraph.get("qas", []):
            question = qa.get("question", "")

            # Extract label using regex for text inside quotes
            match = re.search(r'"([^"]+)"', question)
            if not match:
                continue

            label = match.group(1).strip().lower()

            for answer in qa.get("answers", []):
                raw_text = answer.get("text", "")
                if not raw_text:
                    continue

                text = normalize_clause_text(raw_text)
                if not text:
                    continue

                if len(text.split()) < MIN_WORDS:
                    continue

                text_len = len(text)
                if text_len < MIN_CHAR_LEN or text_len > MAX_CHAR_LEN:
                    continue

                if (text, label) in seen:
                    continue

                seen.add((text, label))
                cleaned_data.append((text, label))

print(f"Extracted {len(cleaned_data)} cleaned clauses.")

Extracted 9915 cleaned clauses.


## STEP 4: Convert to pandas DataFrame
- Columns: "text", "label"

In [37]:
df = pd.DataFrame(cleaned_data, columns=["text", "label"])
df.head()

,text,label
0,The term of this Agreement shall be ten (10) y...,effective date
1,Unless earlier terminated otherwise provided t...,effective date
2,The term of this Agreement shall be ten (10) y...,expiration date
3,If Distributor complies with all of the terms ...,renewal term
4,This Agreement is to be construed according to...,governing law


## STEP 5: Basic analysis
- Print number of samples
- Print number of unique labels
- Show top 10 labels with counts

In [38]:
print(f"Number of samples: {len(df)}")
print(f"Number of unique labels: {df['label'].nunique()}")
print("\nTop 10 labels with counts:")
print(df["label"].value_counts().head(10))

Number of samples: 9915
Number of unique labels: 41

Top 10 labels with counts:
label
license grant                751
cap on liability             665
audit rights                 636
anti-assignment              623
insurance                    546
expiration date              455
governing law                445
post-termination services    442
minimum commitment           414
revenue/profit sharing       410
Name: count, dtype: int64


## STEP 6: Save processed dataset
- Save as CSV to: data/processed/clauses.csv
- Show 5 sample rows from the cleaned dataset

In [39]:
output_path.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(output_path, index=False, encoding="utf-8")
print(f"Saved dataset to {output_path}\n")

df.sample(5, random_state=42)

Saved dataset to d:\Smart_Contract_Review_System\data\processed\clauses.csv



,text,label
2833,The term of this Agreement shall automatically...,effective date
8312,"4.1.9 Ginkgo may elect, at any time in its sol...",termination for convenience
360,The rights granted under such agreements shall...,competitive restriction exception
1261,assign or otherwise transfer this Agreement to...,anti-assignment
3037,Unless otherwise stated in the Appendix the te...,expiration date
